# BirdCLEF+ 2026 - Audio Similarity

Train Soundscapes の各5秒セグメントに対して、Perch埋め込みを用いて最も類似する Train Audio ファイルを検索し、対応表CSVを出力する。

**Input に追加するもの:**
- `birdclef-2026` — コンペデータ
- `google/bird-vocalization-classifier (perch_v2_cpu)` — Perch v2 SavedModel
- `kdmitrie/bc26-tensorflow-2-20-0` — TF 2.20.0 ホイール

In [ ]:
import subprocess, sys
from pathlib import Path

_WHL = Path("/kaggle/input/notebooks/kdmitrie/bc26-tensorflow-2-20-0/wheel")
if not _WHL.exists():
    msg = "Wheel directory not found: " + str(_WHL) + ". Add kdmitrie/bc26-tensorflow-2-20-0 as a Notebook input (not Dataset)."
    raise RuntimeError(msg)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
    str(_WHL / "tensorboard-2.20.0-py3-none-any.whl")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
    str(_WHL / "tensorflow-2.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl")], check=True)
print("Installed TF 2.20.0 from", _WHL)


In [ ]:
# ── Imports ──────────────────────────────────────────────────────
import os, io, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import librosa
import tensorflow as tf
import tensorflow_hub as hub

warnings.filterwarnings("ignore")
print(f"TF: {tf.__version__}")

In [ ]:
# ── Config ───────────────────────────────────────────────────────
N_SAMPLE_PER_SPECIES = 10   # 種ごとにサンプリングする train_audio ファイル数
TOP_K                = 5    # 類似ファイルの上位K件
MAX_AUDIO_SEC        = 30   # train_audio 1ファイルあたりの最大読み込み秒数
SR                   = 32000
SEG_SEC              = 5.0

# Paths
BASE = Path("/kaggle/input/birdclef-2026")
if not BASE.exists():
    BASE = Path("/kaggle/input/competitions/birdclef-2026")

AUDIO_ROOT = BASE / "train_audio"
SC_ROOT    = BASE / "train_soundscapes"
TRAIN_CSV  = BASE / "train.csv"
TAX_CSV    = BASE / "taxonomy.csv"

OUT_CSV = Path("/kaggle/working/audio_similarity.csv")
print(f"BASE: {BASE}")

In [ ]:
# ── Load metadata & sample train_audio ───────────────────────────
train_df = pd.read_csv(TRAIN_CSV)
tax_df   = pd.read_csv(TAX_CSV)

# 種ごとに rating 降順で N_SAMPLE_PER_SPECIES ファイルをサンプリング
sampled = (
    train_df.sort_values("rating", ascending=False)
    .groupby("primary_label")
    .head(N_SAMPLE_PER_SPECIES)
    .reset_index(drop=True)
    .merge(tax_df[["primary_label", "common_name", "scientific_name", "class_name"]],
           on="primary_label", how="left",
           suffixes=("", "_tax"))
)

# common_name は train_df と tax_df どちらか
for col in ["common_name", "scientific_name", "class_name"]:
    if col + "_tax" in sampled.columns:
        sampled[col] = sampled[col].fillna(sampled[col + "_tax"])
        sampled = sampled.drop(columns=[col + "_tax"])

print(f"Sampled: {len(sampled):,} files from {sampled['primary_label'].nunique()} species")
sampled.head(3)

In [ ]:
# ── Load Perch model ─────────────────────────────────────────────
# Kaggle input から SavedModel パスを探す
perch_candidates = [
    "/kaggle/input/bird-vocalization-classifier/google/bird-vocalization-classifier/perch_v2_cpu/1",
    "/kaggle/input/perch-v2-cpu/google/bird-vocalization-classifier/perch_v2_cpu/1",
]
PERCH_PATH = None
for p in perch_candidates:
    if Path(p).exists():
        PERCH_PATH = p
        break
if PERCH_PATH is None:
    # 再帰探索
    found = list(Path("/kaggle/input").rglob("saved_model.pb"))
    if found:
        PERCH_PATH = str(found[0].parent)

print(f"Perch path: {PERCH_PATH}")
model = hub.load(PERCH_PATH)
print("Perch loaded")

In [ ]:
def get_embedding(y_np: np.ndarray) -> np.ndarray:
    """float32 waveform (n_samples,) -> mean-pooled embedding (D,)"""
    with tf.device("/CPU:0"):
        wav = tf.constant(y_np[np.newaxis, :], dtype=tf.float32)
        if hasattr(model, "embed"):
            out = model.embed(wav)
            if isinstance(out, dict):
                emb_key = next((k for k in out if "embed" in k.lower()), None) or list(out.keys())[0]
                emb = out[emb_key].numpy()
            elif isinstance(out, (list, tuple)):
                emb = out[1].numpy()
            else:
                emb = out.numpy()
        elif model.signatures:
            infer = model.signatures[list(model.signatures.keys())[0]]
            in_key = list(infer.structured_input_signature[1].keys())[0]
            out = infer(**{in_key: wav})
            emb_keys = [k for k in out if "embed" in k.lower()]
            key = emb_keys[0] if emb_keys else sorted(out.keys(), key=lambda k: -out[k].shape[-1])[0]
            emb = out[key].numpy()
        else:
            raise ValueError(f"Unknown Perch API. dir={[x for x in dir(model) if not x.startswith(chr(95))]}") 

    if emb.ndim == 3:
        emb = emb[0]
    return emb.mean(axis=0)


def load_audio(path, offset=0.0, duration=None):
    y, _ = librosa.load(str(path), sr=SR, mono=True, offset=offset, duration=duration)
    return y.astype(np.float32)


def pad_or_trim(y, target_samples):
    if len(y) >= target_samples:
        return y[:target_samples]
    return np.pad(y, (0, target_samples - len(y)))


# 動作確認
_test_y = np.zeros(SR * 5, dtype=np.float32)
_emb = get_embedding(_test_y)
print(f"Embedding dim: {_emb.shape[0]}")


In [ ]:
# ── Embed train_audio ────────────────────────────────────────────
train_embeddings = []  # (N_train, D)
train_meta       = []  # list of dict

MAX_SAMPLES = int(MAX_AUDIO_SEC * SR)
SEG_SAMPLES = int(SEG_SEC * SR)

errors = 0
for _, row in tqdm(sampled.iterrows(), total=len(sampled), desc="Embedding train_audio"):
    path = AUDIO_ROOT / row["filename"]
    if not path.exists():
        errors += 1
        continue
    try:
        y = load_audio(path, duration=MAX_AUDIO_SEC)
        y = pad_or_trim(y, SEG_SAMPLES)  # 最初の5秒で代表埋め込みを作成
        emb = get_embedding(y)
        train_embeddings.append(emb)
        train_meta.append({
            "train_filename": str(row["filename"]),
            "primary_label":  row["primary_label"],
            "common_name":    row.get("common_name", ""),
            "scientific_name":row.get("scientific_name", ""),
            "class_name":     row.get("class_name", ""),
            "rating":         row.get("rating", 0),
        })
    except Exception as e:
        errors += 1

train_emb = np.stack(train_embeddings)  # (N_train, D)
# L2 正規化
train_emb_norm = train_emb / (np.linalg.norm(train_emb, axis=1, keepdims=True) + 1e-8)
print(f"Train embeddings: {train_emb_norm.shape}  errors={errors}")

In [ ]:
# ── Embed train_soundscapes (5-sec segments) ──────────────────────
sc_files = sorted(SC_ROOT.glob("*.ogg"))
print(f"Soundscape files: {len(sc_files)}")

sc_embeddings = []  # (N_segs, D)
sc_meta       = []  # list of dict

for sc_path in tqdm(sc_files, desc="Embedding soundscapes"):
    y_full, _ = librosa.load(str(sc_path), sr=SR, mono=True)
    duration  = len(y_full) / SR
    starts    = np.arange(0, duration - SEG_SEC + 0.5, SEG_SEC)  # 5秒刻み

    for start in starts:
        s = int(start * SR)
        e = s + SEG_SAMPLES
        seg = y_full[s:e]
        if len(seg) < SEG_SAMPLES:
            seg = np.pad(seg, (0, SEG_SAMPLES - len(seg)))
        emb = get_embedding(seg.astype(np.float32))
        sc_embeddings.append(emb)
        sc_meta.append({
            "soundscape_file":     sc_path.name,
            "segment_start_sec":   float(start),
            "segment_end_sec":     float(start + SEG_SEC),
        })

sc_emb = np.stack(sc_embeddings)  # (N_segs, D)
sc_emb_norm = sc_emb / (np.linalg.norm(sc_emb, axis=1, keepdims=True) + 1e-8)
print(f"Soundscape embeddings: {sc_emb_norm.shape}")

In [ ]:
# ── Cosine similarity & Top-K ────────────────────────────────────
# sim_matrix: (N_segs, N_train)
sim_matrix = sc_emb_norm @ train_emb_norm.T
top_k_idx  = np.argsort(-sim_matrix, axis=1)[:, :TOP_K]  # (N_segs, K)

# 結果を DataFrame に変換
rows = []
for seg_i, meta in enumerate(sc_meta):
    for rank, train_i in enumerate(top_k_idx[seg_i], start=1):
        row = {**meta, **train_meta[train_i],
               "rank": rank,
               "cosine_sim": float(sim_matrix[seg_i, train_i])}
        rows.append(row)

result_df = pd.DataFrame(rows)
col_order = [
    "soundscape_file", "segment_start_sec", "segment_end_sec",
    "rank", "train_filename", "primary_label",
    "common_name", "scientific_name", "class_name", "rating", "cosine_sim"
]
result_df = result_df[col_order]
print(f"Result rows: {len(result_df):,}")
result_df.head(10)

In [ ]:
# ── Save ─────────────────────────────────────────────────────────
result_df.to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV}  ({len(result_df):,} rows)")

# 簡易確認
n_segs = result_df[["soundscape_file", "segment_start_sec"]].drop_duplicates()
print(f"  Soundscape files:   {result_df['soundscape_file'].nunique()}")
print(f"  Total segments:     {len(n_segs):,}")
print(f"  Top-K per segment:  {TOP_K}")
print("\nTop similarity examples:")
print(result_df[result_df["rank"] == 1].nlargest(5, "cosine_sim")[
    ["soundscape_file", "segment_start_sec", "common_name", "class_name", "cosine_sim"]
])